# Практика 10. Неперервні розподіли в `scipy.stats`

**Студент:** Гузь Юрій  
**Група:** ІТ-41  
**Варіант:** 3, Одеса  
**Липнева середньомісячна температура:** 23 °C

Мета: згенерувати вибірку денних температур Одеси, підібрати нормальну модель за середнім і стандартним відхиленням вибірки та застосувати функції `pdf()`, `cdf()` і `ppf()`.

## Завдання 1. Вибірка денних температур

Для варіанта 3 липневе середнє Одеси дорівнює 23 °C. Фіксоване початкове значення генератора забезпечує відтворюваність результату.

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

np.random.seed(3)
daily_temps = np.random.normal(loc=23, scale=2.5, size=30)

print("Денні температури (°C):")
print(daily_temps)
print(f"Середнє вибірки: {daily_temps.mean():.2f} °C")
print(f"Стандартне відхилення вибірки: {daily_temps.std():.2f} °C")

Отримане середнє має бути близьким до заданих 23 °C, але не обов'язково збігатися з ним: окремі денні значення містять випадкові відхилення.

## Завдання 2. Параметри нормального розподілу

Параметри моделі оцінюємо безпосередньо за згенерованою вибіркою. `std()` NumPy за замовчуванням обчислює стандартне відхилення з `ddof=0`.

In [ ]:
mean = daily_temps.mean()
std = daily_temps.std()
dist = stats.norm(loc=mean, scale=std)

print(f"Параметри підібраного нормального розподілу: μ = {mean:.2f} °C, σ = {std:.2f} °C")

## Завдання 3. Гістограма і теоретична PDF-крива

`density=True` переводить гістограму в густину, тому її можна коректно порівнювати з `pdf()`, яка теж повертає густину.

In [ ]:
x = np.linspace(daily_temps.min() - 2, daily_temps.max() + 2, 300)

plt.figure(figsize=(8, 5))
plt.hist(daily_temps, bins=8, density=True, edgecolor="white", alpha=0.7, label="Денні температури")
plt.plot(x, dist.pdf(x), color="crimson", linewidth=2, label="Нормальна модель: PDF")
plt.xlabel("Температура, °C")
plt.ylabel("Густина")
plt.title("Температури Одеси: гістограма та нормальна модель")
plt.legend()
plt.show()

Оцініть графік: теоретична крива має показувати загальну дзвоноподібну форму й проходити поблизу висот стовпців, але стовпці можуть помітно коливатися навколо неї. Вибірка містить лише 30 випадкових спостережень, а результат залежить і від випадкової реалізації, і від обраної кількості інтервалів гістограми.

## Завдання 4. Ймовірність у межах одного стандартного відхилення

Різниця значень CDF на верхній і нижній межах дає ймовірність потрапляння в інтервал `[mean - std, mean + std]`.

In [ ]:
lower = mean - std
upper = mean + std
p_within_one_std = dist.cdf(upper) - dist.cdf(lower)

print(f"Інтервал: [{lower:.2f}, {upper:.2f}] °C")
print(f"Модельна ймовірність: {p_within_one_std:.4f} ({p_within_one_std:.2%})")
print(f"Різниця з 68.27%: {abs(p_within_one_std - 0.6827) * 100:.4f} відсоткового пункту")

Для нормального розподілу правило 68–95–99.7 передбачає близько 68.27% значень у межах одного стандартного відхилення від середнього. Тут `cdf()` рахує ймовірність саме підібраної нормальної моделі, а межі задані як `mean ± std` цієї ж моделі, тому результат має бути практично 68.27%. Це не емпірична частка температур у скінченній вибірці: у 30 конкретних спостереженнях фактична частка може відрізнятися, що видно з гістограми.

## Завдання 5. 95-й процентиль через PPF

`ppf(0.95)` знаходить температуру, нижче якої за підібраною моделлю лежить 95% значень.

In [ ]:
percentile_95 = dist.ppf(0.95)
print(f"95-й процентиль: {percentile_95:.2f} °C")
print(f"Інтерпретація: за моделлю близько 95% денних температур нижче {percentile_95:.2f} °C, а близько 5% — вище.")

## Письмові відповіді

1. **Чим відрізняються `pdf(x)` і `cdf(x)`?** `pdf(x)` повертає значення густини й описує висоту кривої біля точки `x`; це не ймовірність отримати рівно це значення. `cdf(x)` повертає накопичену ймовірність `P(X ≤ x)` — частку розподілу ліворуч від `x`. Ймовірність інтервалу знаходять як різницю CDF на його межах.

2. **Чому `pdf(x)` може бути більшим за 1?** PDF є густиною, а не ймовірністю, тому її значення може перевищувати 1, якщо розподіл достатньо зосереджений на вузькому інтервалі. Імовірність отримують площею під кривою на інтервалі; повна площа під PDF дорівнює 1, тож обмеження на ймовірність не порушується.

3. **Чому `ppf()` є оберненою до `cdf()`?** `cdf(x)` приймає значення змінної `x` і повертає накопичену ймовірність `p`. `ppf(p)` приймає ймовірність `p` і повертає відповідне значення `x` (квантиль), для якого `cdf(x) = p`.

## Підсумок

За згенерованими денними температурами Одеси оцінено параметри нормальної моделі, побудовано гістограму з `density=True` і накладено PDF-криву. CDF використано для ймовірності температурного інтервалу, а PPF — для знаходження 95-го процентиля.